In [4]:
import pandas as pd
import rasterio
from geollm_scripts.geollm_utils import *
import re
from geollm_scripts.make_predictions_and_visualize import run_task_for_data

path_raster = "data/geollm/ppp_2020_1km_Aggregated.tif"
path_prompts = "data/geollm/world_prompts.jsonl"    

In [26]:
def extract_coord_from_line(text):
    pattern = r'Coordinates:\s*\((-?\d+\.\d+),\s*(-?\d+\.\d+)\)'

    match = re.search(pattern, text)
    if match:
        lon = float(match.group(1))
        lat = float(match.group(2))
    
    return lon, lat

def get_density_at(src, lon, lat):
    return list(src.sample([(lon, lat)]))[0][0]

def get_coord_from_json(json_file, line_range=(0, 10)):
    prompts = []
    lons = []
    lats = []
    df_json = pd.read_json(json_file, lines=True)
    for i in range(*line_range):
        prompt = df_json.iloc[i]['text']
        lon, lat = get_coordinates(prompt)

        prompts.append(prompt)
        lons.append(lon)
        lats.append(lat)
    
    df_ret = pd.DataFrame({"prompt" : prompts, "lon": lons, "lat" : lats})
    return df_ret

def add_population_density(df, tif_path, lon_col="lon", lat_col="lat"):

    coords = list(zip(df[lon_col], df[lat_col]))

    groundtruth = [extract_data(lat, lon, tif_path) for lat, lon in coords]

    df = df.copy()
    df["population"] = groundtruth

    return df

In [27]:
df = get_coord_from_json(path_prompts, line_range=(0,2000))
print(df.shape)
df.head()

(2000, 3)


,prompt,lon,lat
0,"Coordinates: (-5.87958, 14.98625)\n\nAddress: ...",-5.87958,14.98625
1,"Coordinates: (67.29542, 14.72792)\n\nAddress: ...",67.29542,14.72792
2,"Coordinates: (58.37875, -134.63042)\n\nAddress...",58.37875,-134.63042
3,"Coordinates: (55.05375, 58.97792)\n\nAddress: ...",55.05375,58.97792
4,"Coordinates: (53.18708, 8.23625)\n\nAddress: ""...",53.18708,8.23625


In [28]:
df = add_population_density(df, path_raster)
print(df.shape)
df.head()

(2000, 4)


,prompt,lon,lat,population
0,"Coordinates: (-5.87958, 14.98625)\n\nAddress: ...",-5.87958,14.98625,44509.261719
1,"Coordinates: (67.29542, 14.72792)\n\nAddress: ...",67.29542,14.72792,11661.978516
2,"Coordinates: (58.37875, -134.63042)\n\nAddress...",58.37875,-134.63042,17762.666016
3,"Coordinates: (55.05375, 58.97792)\n\nAddress: ...",55.05375,58.97792,33767.539062
4,"Coordinates: (53.18708, 8.23625)\n\nAddress: ""...",53.18708,8.23625,193455.515625


In [2]:
model_name = "HuggingFaceTB/SmolLM3-3B"
device = "cuda"
api = "local"
prompt_file_path = "data/geollm/world_prompts.jsonl"
less_path = "data/geollm/small_world_prompts.jsonl"
task = "Population Density"

In [3]:

run_task_for_data(model_api=api, model=model_name, task=task, prompt_file_path=less_path, api_key=device)


 >> Loading local model << 


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PROMPT 1:

You will be given data about a specific location randomly sampled from all human-populated locations on Earth.
You give your rating keeping in mind that it is relative to all other human-populated locations on Earth (from all continents, countries, etc.).
You provide ONLY your answer in the exact format "My answer is X.X." where 'X.X' represents your rating for the given topic.


Coordinates: (-5.87958, 14.98625)

Address: "Uíge, Angola"

Nearby Places:
"
3.4 km North-East: Kimbala
4.7 km North-East: Kimpangu
4.8 km North-East: Kimvindu
5.9 km North-East: Kikongo
15.7 km North-West: Tunda
15.9 km North-East: Kungu Lusanga
17.5 km North: Luzanga
18.2 km North: Mbanza Ngombe
18.4 km North-West: Wene
18.6 km North-West: Kongo Dia Kati
"

Population Density (On a Scale from 0.0 to 9.9):

COMPLETION: <think>

</think>
My answer is 1.5.

RATING: 1.5

PROMPT 2:

You will be given data about a specific location randomly sampled from all human-populated locations on Earth.
You give y

In [2]:
from geollm_scripts.calculate_spearman_correlation import calculate_spearman_correlation
import pandas as pd

def print_spearman_correl(predictions_csv, groundtruth_tif):
    df = pd.read_csv(predictions_csv)
    if 'Latitude' in df.columns and 'Longitude' in df.columns and 'Predictions' in df.columns:
        coordinates = list(zip(df['Latitude'], df['Longitude']))
        predictions = df['Predictions']
    else:
        raise ValueError("CSV file must contain 'Latitude', 'Longitude', and 'Predictions' columns.")

    corr = calculate_spearman_correlation(coordinates, predictions, groundtruth_tif)

    print(f"Spearman correlation: {corr:.2f}")

In [3]:
predictions_csv = "results/HuggingFaceTB_SmolLM3_3B_Population_Density_small_world_prompts.csv"
groundtruth_tif = "data/geollm/ppp_2020_1km_Aggregated.tif"

print_spearman_correl(predictions_csv, groundtruth_tif)

Spearman correlation: 0.22


## GeoLLM sur les couches cachés